# Session 1: Modern LLM Foundations for Agentic Systems (Student Notebook)

Welcome to Session 1! In this notebook, you will explore the foundational concepts behind Large Language Models (LLMs) and learn how to interact with them programmatically. These skills form the bedrock for building agentic AI systems in subsequent sessions.

## Learning Objectives

By the end of this session, you will be able to:

1. **Set up and configure** LLM API connections using the OpenAI Python SDK
2. **Understand tokenization** and how context windows affect LLM behavior
3. **Control model output** by tuning parameters such as temperature, top_p, and max_tokens
4. **Use system messages** to shape LLM response behavior and persona
5. **Build basic LLM pipelines** by chaining multiple API calls together
6. **Lay the groundwork** for conversational agents with message history management
7. **Generate and compare embeddings** to understand vector representations of text
8. **Visualize embeddings** using dimensionality reduction and inspect semantic similarity
9. **Understand how LLMs support reasoning** and agent behavior through emergent capabilities

In [1]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# ============================================================
# Imports and Setup
# ============================================================

import openai
import tiktoken
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# Ensure your OpenAI API key is set as an environment variable:
#   export OPENAI_API_KEY="sk-..."
# Or uncomment the line below and paste your key (not recommended for production):
# os.environ["OPENAI_API_KEY"] = "sk-..."

print("All imports successful!")
print(f"OpenAI SDK version: {openai.__version__}")

All imports successful!
OpenAI SDK version: 2.31.0


---
## Demos (Follow Along)

The following five demos are fully coded. Run each cell, observe the output, and make sure you understand what is happening before moving on.

### Demo 1: Setting Up LLM API Connections

In this demo we initialize the OpenAI client and make our first chat completion call. The `client` object handles authentication, request formatting, and response parsing for us.

**Scenario:** A McKinsey team needs to quickly query an LLM to summarize industry concepts for a client engagement kickoff.

In [2]:
# Demo 1 - Setting Up LLM API Connections

# Step 1: Initialize the OpenAI client
# The client automatically reads OPENAI_API_KEY from the environment.
client = openai.OpenAI()

# Step 2: Make a simple chat completion call
# Scenario: A McKinsey analyst needs a quick definition for a client deck
response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "user", "content": "What is a digital transformation strategy? Explain in two sentences for a McKinsey client presentation."}
    ],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "100"))
)

# Step 3: Print the response
print("Model used :", response.model)
print("Finish reason:", response.choices[0].finish_reason)
print("\nAssistant response:")
print(response.choices[0].message.content)

Model used : gpt-4o-mini-2024-07-18
Finish reason: stop

Assistant response:
A digital transformation strategy is a comprehensive plan that organizations adopt to leverage digital technologies, enhance operational efficiency, and improve customer experiences. By integrating digital solutions across all aspects of the business, companies can drive innovation, adapt to market changes, and achieve sustainable growth.


### Demo 2: Understanding Tokenization and Context Windows

Tokens are the fundamental units that LLMs process. A token can be a word, part of a word, or even a single character. Understanding tokenization helps you:
- Estimate API costs (billing is per-token)
- Stay within context window limits
- Write more efficient prompts

In [3]:
# Demo 2 - Understanding Tokenization and Context Windows

# Step 1: Get the tokenizer for gpt-4o-mini
encoder = tiktoken.encoding_for_model("gpt-4o-mini")

# Step 2: Encode various consulting-related texts and count tokens
texts = [
    "McKinsey & Company",
    "Digital transformation is reshaping how Fortune 500 companies create value across their organizations.",
    "The client's operating model requires restructuring to achieve the target $200M in synergies.",
    "Organizational restructuring and operational efficiency optimization",
    "def calculate_market_size(tam, penetration_rate):\n    sam = tam * penetration_rate\n    return sam * 0.15  # McKinsey target share"
]

print("Token counts for different consulting texts:")
print("=" * 60)
for text in texts:
    tokens = encoder.encode(text)
    print(f"\nText   : {text[:60]}{'...' if len(text) > 60 else ''}")
    print(f"Chars  : {len(text)}")
    print(f"Tokens : {len(tokens)}")
    print(f"Token IDs (first 10): {tokens[:10]}")

# Step 3: Demonstrate context window limits
print("\n" + "=" * 60)
print("Context window reference:")
print("  gpt-4o-mini : 128,000 tokens")
print("  gpt-4o      : 128,000 tokens")
print("  gpt-3.5-turbo: 16,385 tokens")

# Simulate a long due diligence report consuming the window
long_text = "The target company operates in a fragmented market with strong growth potential. " * 500
long_tokens = encoder.encode(long_text)
print(f"\nLong due diligence excerpt ({len(long_text)} chars) = {len(long_tokens)} tokens")
print(f"That is {len(long_tokens)/128000*100:.2f}% of the gpt-4o-mini context window.")

Token counts for different consulting texts:

Text   : McKinsey & Company
Chars  : 18
Tokens : 5
Token IDs (first 10): [41691, 73519, 28230, 827, 9709]

Text   : Digital transformation is reshaping how Fortune 500 companie...
Chars  : 102
Tokens : 16
Token IDs (first 10): [44057, 26242, 382, 132104, 19327, 1495, 57526, 220, 3234, 6005]

Text   : The client's operating model requires restructuring to achie...
Chars  : 93
Tokens : 17
Token IDs (first 10): [976, 122281, 14359, 2359, 10860, 116848, 316, 12998, 290, 3783]

Text   : Organizational restructuring and operational efficiency opti...
Chars  : 68
Tokens : 8
Token IDs (first 10): [19888, 482, 1953, 116848, 326, 33216, 17456, 34658]

Text   : def calculate_market_size(tam, penetration_rate):
    sam = ...
Chars  : 129
Tokens : 33
Token IDs (first 10): [1314, 17950, 165999, 5044, 2025, 313, 11, 86834, 20477, 1883]

Context window reference:
  gpt-4o-mini : 128,000 tokens
  gpt-4o      : 128,000 tokens
  gpt-3.5-turbo: 16,385 tokens



### Demo 3: Exploring Model Parameters

LLMs expose several parameters that control the randomness and length of generated text:

| Parameter | Description |
|-----------|-------------|
| `temperature` | Controls randomness. 0 = deterministic, 1 = creative |
| `top_p` | Nucleus sampling. Lower values = more focused output |
| `max_tokens` | Maximum number of tokens in the response |

In [4]:
# Demo 3 - Exploring Model Parameters

client = openai.OpenAI()
prompt = "Write a one-sentence executive summary of generative AI's impact on the management consulting industry."

# --- Temperature comparison ---
print("TEMPERATURE COMPARISON")
print("=" * 60)

for temp in [0.0, 0.5, 1.0]:
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "80"))
    )
    print(f"\nTemperature {temp}:")
    print(f"  {response.choices[0].message.content}")

# --- top_p comparison ---
print("\n" + "=" * 60)
print("TOP_P COMPARISON")
print("=" * 60)

for top_p in [0.1, 0.5, 1.0]:
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[{"role": "user", "content": prompt}],
        top_p=top_p,
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "80"))
    )
    print(f"\ntop_p {top_p}:")
    print(f"  {response.choices[0].message.content}")

# --- max_tokens comparison ---
print("\n" + "=" * 60)
print("MAX_TOKENS COMPARISON")
print("=" * 60)

for max_tok in [10, 30, 100]:
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[{"role": "user", "content": prompt}],
        temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")),
        max_tokens=max_tok
    )
    text = response.choices[0].message.content
    print(f"\nmax_tokens={max_tok} (finish_reason={response.choices[0].finish_reason}):")
    print(f"  {text}")

TEMPERATURE COMPARISON

Temperature 0.0:
  Generative AI is revolutionizing the management consulting industry by enhancing data analysis, streamlining report generation, and enabling more personalized client solutions, ultimately driving efficiency and innovation in service delivery.

Temperature 0.5:
  Generative AI is revolutionizing the management consulting industry by enhancing data analysis, automating routine tasks, and enabling more innovative problem-solving, thereby increasing efficiency and driving value for clients.

Temperature 1.0:
  Generative AI is transforming the management consulting industry by enhancing data analysis, streamlining report generation, and enabling more personalized client solutions, thereby increasing efficiency and driving innovative strategic insights.

TOP_P COMPARISON

top_p 0.1:
  Generative AI is revolutionizing the management consulting industry by enhancing data analysis, streamlining report generation, and enabling more personalized client 

### Demo 4: Comparing LLM Response Behaviors

The **system message** is a powerful lever for controlling how the LLM behaves. By changing the system message, you can make the same model act as different McKinsey expert "personas," each with its own tone, expertise, and communication style — mirroring how different practice areas approach the same client question.

In [5]:
# Demo 4 - Comparing LLM Response Behaviors

client = openai.OpenAI()

user_question = "Our retail client is experiencing a 15% decline in same-store sales. What should we investigate?"

personas = {
    "McKinsey Partner (Strategy)": (
        "You are a McKinsey Senior Partner leading the Strategy & Corporate Finance practice. "
        "You think in terms of competitive dynamics, portfolio strategy, and value creation. "
        "Use structured, MECE frameworks and speak with authority."
    ),
    "McKinsey Associate (Operations)": (
        "You are a McKinsey Associate in the Operations practice. You focus on supply chain, "
        "store-level performance metrics, and operational efficiency. Be data-driven and suggest "
        "specific analyses to run."
    ),
    "McKinsey Expert (Digital & Analytics)": (
        "You are a McKinsey Digital & Analytics expert. You focus on digital channels, "
        "customer analytics, omnichannel strategy, and technology enablement. "
        "Recommend data-driven approaches and digital solutions."
    ),
}

print(f"Client Question: {user_question}")
print("=" * 60)

for persona_name, system_msg in personas.items():
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_question}
        ],
        temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.7")),
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "200"))
    )
    print(f"\n--- {persona_name} ---")
    print(response.choices[0].message.content)
    print()

Client Question: Our retail client is experiencing a 15% decline in same-store sales. What should we investigate?

--- McKinsey Partner (Strategy) ---
To address the 15% decline in same-store sales for your retail client, we should adopt a structured approach using a MECE (Mutually Exclusive, Collectively Exhaustive) framework. This will help us identify the root causes and potential solutions systematically. We can categorize our investigation into three main areas: **Market Dynamics, Customer Behavior, and Operational Factors**.

### 1. Market Dynamics
   - **Competitive Landscape**: 
     - Analyze competitors’ performance and strategies. Are they gaining market share? What promotions or innovations are they implementing?
     - Assess any new entrants or disruptive business models affecting the market.
   - **Economic Environment**: 
     - Evaluate macroeconomic factors such as consumer confidence, unemployment rates, and inflation that may impact spending.
     - Investigate regi

### Demo 5: Building a Basic LLM Pipeline

A pipeline chains multiple LLM calls together, where the output of one step becomes the input of the next. This pattern is fundamental to agentic systems and mirrors how McKinsey structures analysis — first synthesize raw research, then extract structured insights for client delivery.

In [6]:
# Demo 5 - Building a Basic LLM Pipeline

client = openai.OpenAI()

def call_llm(system_message, user_message, temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")), max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))):
    """Helper function to make an LLM call with a system and user message."""
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content


# --- Pipeline: Summarize Industry Research -> Extract Key Insights ---

industry_research = """
The global management consulting market is undergoing a fundamental transformation driven by 
generative AI and advanced analytics. McKinsey's own research indicates that up to 70% of 
business activities could be automated with current technology. Traditional strategy engagements 
that once required weeks of analyst research can now be augmented with AI-driven market analysis 
in days. However, the human element remains critical — clients value the judgment, relationship 
management, and change leadership that experienced consultants provide. The shift is creating 
new service lines around AI implementation, responsible AI governance, and digital capability 
building. Firms that successfully integrate AI into their delivery model are seeing 30-40% 
improvements in engagement efficiency while maintaining or improving quality scores. The key 
challenges include managing data confidentiality across client engagements, ensuring AI outputs 
meet McKinsey's quality standards, building consultant capabilities in prompt engineering and 
AI-augmented analysis, and maintaining the trusted advisor relationship in an AI-augmented world.
"""

print("PIPELINE STEP 1: Summarize Industry Research")
print("=" * 60)

summary = call_llm(
    system_message="You are a McKinsey research analyst. Summarize the given text in 2-3 sentences suitable for a partner briefing.",
    user_message=f"Summarize this industry research:\n\n{industry_research}"
)
print(summary)

print("\nPIPELINE STEP 2: Extract Key Insights for Client Deck")
print("=" * 60)

key_points = call_llm(
    system_message="You are a McKinsey engagement manager preparing a client presentation. Extract exactly 5 key insights as a numbered list, each suitable for a slide bullet point.",
    user_message=f"Extract the key insights from this summary:\n\n{summary}"
)
print(key_points)

print("\nPIPELINE RESULT: Complete output")
print("=" * 60)
print(f"Original research : {len(industry_research.split())} words")
print(f"Summary length    : {len(summary.split())} words")
print(f"\nKey Insights for Client Deck:\n{key_points}")

PIPELINE STEP 1: Summarize Industry Research
The global management consulting market is rapidly evolving due to generative AI and advanced analytics, with McKinsey's research suggesting that up to 70% of business activities could be automated. While AI enhances efficiency—improving engagement by 30-40%—the human element remains essential for client relationships and change leadership. Key challenges include data confidentiality, maintaining quality standards, and developing consultant skills in AI integration while preserving the trusted advisor role.

PIPELINE STEP 2: Extract Key Insights for Client Deck
1. The global management consulting market is evolving, with generative AI and advanced analytics poised to automate up to 70% of business activities.

2. AI implementation can enhance efficiency, leading to a 30-40% improvement in client engagement.

3. Despite technological advancements, the human element is crucial for maintaining strong client relationships and effective change le

### Demo 6: Embeddings and Vector Representations

Embeddings convert text into dense numerical vectors that capture semantic meaning. Two sentences with similar meaning will have similar embedding vectors, even if they use different words. This is the foundation for:
- Semantic search across McKinsey knowledge bases and past engagement documents
- Clustering client industries and market research by topic
- Similarity-based retrieval for RAG-powered consulting assistants

### Task 1: Configure and Test Multiple LLM Provider Connections

Create a function that initializes an OpenAI client, sends a McKinsey-relevant test message, and returns the response. This verifies that your API key and network connection are working correctly.

In [7]:
# Task 1 - Configure and Test Multiple LLM Provider Connections

# TODO: Create a function that initializes an OpenAI client and tests the connection
# by sending a McKinsey-relevant message and returning the response.
#
# Hint: Use openai.OpenAI() to create the client
# Hint: Use client.chat.completions.create() with model="gpt-4o-mini"
# Hint: Try a consulting-relevant prompt like "Define 'MECE' as used in McKinsey problem-solving."
# Hint: Return response.choices[0].message.content

def test_llm_connection():
    """Initialize OpenAI client and test with a consulting-relevant message."""
    # YOUR CODE HERE
    pass


# Test your function
# result = test_llm_connection()
# print(result)

### Task 2: Analyze Token Usage and Optimize Prompt Length

Write a function that counts the tokens in a consulting research brief and, if the token count exceeds a given limit, truncates the prompt intelligently at sentence boundaries so it fits within the budget. This is a common challenge when feeding long client documents into LLMs.

In [8]:
# Task 2 - Analyze Token Usage and Optimize Prompt Length

# TODO: Write a function that takes a prompt string, counts its tokens using tiktoken,
# and if it exceeds a given limit, truncates it intelligently (at sentence boundaries).
#
# Hint: Use tiktoken.encoding_for_model("gpt-4o-mini") to get the encoder
# Hint: Use encoder.encode(text) to get tokens
# Hint: Split by sentences (e.g., split on ". ") and rebuild until under the limit

def optimize_prompt(prompt, max_tokens=500):
    """Count tokens and truncate at sentence boundaries if over the limit."""
    # YOUR CODE HERE
    pass


# Test with a long McKinsey market research brief
# long_prompt = (
#     "The global healthcare market is projected to reach $12 trillion by 2028. "
#     "Digital health solutions are growing at 25% CAGR across developed markets. "
# ) * 200
# optimized = optimize_prompt(long_prompt, max_tokens=50)
# print(optimized)

### Task 3: Experiment with Model Parameters

Create a function that sends the same McKinsey-relevant prompt at three different temperature settings and returns all three responses so you can compare how randomness affects the quality and creativity of consulting outputs.

In [9]:
# Task 3 - Experiment with Model Parameters

# TODO: Create a function that sends the same prompt with 3 different temperature
# settings (0.0, 0.5, 1.0) and returns all three responses for comparison.
#
# Hint: Use a loop over temperature values [0.0, 0.5, 1.0]
# Hint: Store results in a dictionary {temperature: response_text}
# Hint: Use the same prompt and model for fair comparison

def compare_temperatures(prompt):
    """Send the same prompt at different temperatures and return all responses."""
    # YOUR CODE HERE
    pass


# Test with a McKinsey-relevant prompt
# results = compare_temperatures("Describe the competitive advantages of a vertically integrated supply chain for a consumer goods company.")
# for temp, text in results.items():
#     print(f"Temperature {temp}: {text}\n")

### Task 4: Build a Simple Conversational Agent Foundation

Build a `SimpleAgent` class that acts as a McKinsey engagement manager assistant. It should maintain a conversation history, use a system message to define its consulting persona, and hold multi-turn conversations where it remembers prior context (e.g., the client's industry).

In [10]:
# Task 4 - Build a Simple Conversational Agent Foundation

# TODO: Build a McKinsey engagement manager assistant that:
# 1. Maintains a message history list
# 2. Has a system message defining it as a McKinsey engagement manager
#    (uses MECE frameworks, data-driven, professional consulting tone)
# 3. Takes user input, appends to history, sends to LLM, appends response
# 4. Returns the assistant's response
#
# Hint: Start with messages = [{"role": "system", "content": "You are a McKinsey engagement manager..."}]
# Hint: Append user messages with {"role": "user", "content": user_input}
# Hint: After getting response, append {"role": "assistant", "content": response}

class SimpleAgent:
    def __init__(self):
        """Initialize the agent with a McKinsey consulting persona."""
        # YOUR CODE HERE
        pass

    def chat(self, user_input):
        """Process a user message and return the assistant's response."""
        # YOUR CODE HERE
        pass

    def get_history(self):
        """Return the full conversation history."""
        # YOUR CODE HERE
        pass


# Test your agent with a multi-turn McKinsey conversation
# agent = SimpleAgent()
# print(agent.chat("We have a new client in the automotive sector facing margin pressure. What should we focus on?"))
# print("---")
# print(agent.chat("Which of those areas would have the quickest impact?"))
# print("---")
# print(f"History length: {len(agent.get_history())} messages")

### Task 5: Generate and Compare Embeddings for Semantic Search

Create a function that generates embeddings for a set of consulting-related phrases using the OpenAI Embeddings API, then computes cosine similarity between all pairs to find semantically similar phrases. This is the foundation for building semantic search over McKinsey knowledge bases.

In [11]:
# Task 5 - Generate and Compare Embeddings for Semantic Search

# TODO: Create two functions:
# 1. get_embeddings(texts) - takes a list of strings, returns a dict {text: embedding_vector}
# 2. cosine_similarity(vec_a, vec_b) - computes cosine similarity between two vectors
#
# Then generate embeddings for the consulting phrases below and find the most similar pairs.
#
# Hint: Use client.embeddings.create(model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=texts)
# Hint: Each item in response.data has an .embedding attribute (a list of floats)
# Hint: Cosine similarity = dot(a, b) / (norm(a) * norm(b)) -- use np.dot and np.linalg.norm
# Hint: Pairs with similarity > 0.7 are semantically related

def get_embeddings(texts):
    """Generate embeddings for a list of texts using the OpenAI API."""
    # YOUR CODE HERE
    pass


def cosine_similarity(vec_a, vec_b):
    """Compute cosine similarity between two vectors."""
    # YOUR CODE HERE
    pass


# Test with these McKinsey consulting phrases
# consulting_phrases = [
#     "Revenue growth decelerated in Q3 due to market headwinds",
#     "Sales momentum slowed in the third quarter",
#     "The company needs to restructure its operating model",
#     "Post-merger integration is behind schedule",
#     "Customer acquisition costs are rising faster than lifetime value",
#     "The cost of acquiring new customers exceeds their long-term profitability",
# ]
# embeddings = get_embeddings(consulting_phrases)
# for i, a in enumerate(consulting_phrases):
#     for j, b in enumerate(consulting_phrases):
#         if j > i:
#             sim = cosine_similarity(embeddings[a], embeddings[b])
#             if sim > 0.7:
#                 print(f"Similarity {sim:.3f}: '{a[:50]}' <-> '{b[:50]}'")

### Task 6: Build a Multi-Step Analysis Pipeline with Quality Review

Build a 3-step consulting analysis pipeline that chains LLM calls: (1) an Associate analyzes the problem, (2) an Engagement Manager generates recommendations based on the analysis, (3) a Senior Partner reviews the work for quality. This implements the **self-reflection pattern** where the LLM critiques its own prior output.

In [12]:
# Task 6 - Build a Multi-Step Analysis Pipeline with Quality Review

# TODO: Create a function analysis_pipeline(client_problem) that chains 3 LLM calls:
#   Step 1 (Associate): Analyze the problem using MECE framework
#   Step 2 (EM): Generate 3 recommendations based on Step 1 output
#   Step 3 (Partner): Review Steps 1+2 for completeness, rigor, and actionability
#
# Hint: Use different system messages for each step (different personas)
# Hint: Pass the output of each step as input to the next via the user message
# Hint: Step 3 should receive BOTH the analysis and recommendations for review
# Hint: Return a dict with keys: "analysis", "recommendations", "review"

def analysis_pipeline(client_problem):
    """Run a 3-step consulting analysis pipeline with quality review."""
    client = openai.OpenAI()
    # YOUR CODE HERE
    pass


# Test the pipeline
# problem = "A mid-size European bank is losing market share in retail deposits to digital-first neobanks. Their NPS has dropped 20 points in 18 months."
# result = analysis_pipeline(problem)
# print("STEP 1: Analysis\n", result["analysis"])
# print("\nSTEP 2: Recommendations\n", result["recommendations"])
# print("\nSTEP 3: Quality Review\n", result["review"])

### Task 7: Estimate API Costs from Token Usage

Create a function that sends a prompt to the LLM, extracts the token usage from the response, and calculates the estimated cost based on OpenAI's pricing. This is essential for budgeting AI-augmented consulting engagements.

In [13]:
# Task 7 - Estimate API Costs from Token Usage

# TODO: Create a function estimate_cost(prompt, system_message, model) that:
#   1. Makes an API call and gets the response
#   2. Extracts token usage from response.usage (prompt_tokens, completion_tokens)
#   3. Calculates cost using per-token pricing
#   4. Returns response text, token counts, and estimated cost
#
# Hint: response.usage has .prompt_tokens and .completion_tokens attributes
# Hint: gpt-4o-mini pricing: $0.15 per 1M input tokens, $0.60 per 1M output tokens
# Hint: Cost = (input_tokens * input_price / 1_000_000) + (output_tokens * output_price / 1_000_000)
# Hint: Return a dict with keys: response, input_tokens, output_tokens, estimated_cost_usd

def estimate_cost(prompt, system_message="You are a helpful assistant.", model=None):
    """Send a prompt and return the response with token usage and cost estimate."""
    # YOUR CODE HERE
    pass


# Test with a consulting prompt
# result = estimate_cost(
#     prompt="Outline the key steps for commercial due diligence in a SaaS acquisition.",
#     system_message="You are a McKinsey M&A due diligence specialist."
# )
# print(f"Input tokens: {result['input_tokens']}")
# print(f"Output tokens: {result['output_tokens']}")
# print(f"Estimated cost: ${result['estimated_cost_usd']:.6f}")
# print(f"Projected cost for 500 queries: ${result['estimated_cost_usd'] * 500:.4f}")

### Task 8: Build a Context-Aware Routing Agent

Build an agent that uses a two-stage approach: first, a "router" LLM call classifies the user's question into a McKinsey practice area (Strategy, Operations, Digital, or M&A). Then, a second LLM call uses the appropriate practice-specific system message to generate an expert response. This **classify-then-route** pattern is fundamental to multi-agent architectures.

In [14]:
# Task 8 - Build a Context-Aware Routing Agent

# TODO: Build a route_and_respond(question) function that:
#   1. Uses an LLM call to classify the question into: strategy, operations, digital, or m&a
#   2. Selects the matching practice-specific system message
#   3. Uses a second LLM call with that persona to generate an expert response
#   4. Returns {"practice_area": ..., "expert_response": ...}
#
# Hint: For the router, use a system message like: "Classify into one practice area. Respond with ONLY one word: strategy, operations, digital, or m&a"
# Hint: Use temperature=0 for the router (deterministic classification)
# Hint: Use max_tokens=5 for the router (only need one word)
# Hint: Create a dict mapping practice names to system messages, then use .get() with a default

PRACTICE_PERSONAS = {
    "strategy": "You are a McKinsey Strategy expert. Focus on competitive positioning, market entry, and value creation.",
    "operations": "You are a McKinsey Operations expert. Focus on supply chain, lean processes, and operational KPIs.",
    "digital": "You are a McKinsey Digital expert. Focus on digital transformation, AI/ML use cases, and data strategy.",
    "m&a": "You are a McKinsey M&A specialist. Focus on deal structuring, synergies, and integration planning."
}

def route_and_respond(question):
    """Classify the question into a practice area, then respond with a specialist persona."""
    # YOUR CODE HERE
    pass


# Test with different types of questions
# questions = [
#     "How should we evaluate entering the Indian EV market?",
#     "Our warehouse fulfillment time is 40% above benchmark. How do we fix this?",
#     "What AI use cases should a retail bank prioritize?",
#     "We're acquiring a competitor for $2B. How do we estimate synergies?"
# ]
# for q in questions:
#     result = route_and_respond(q)
#     print(f"Q: {q}")
#     print(f"Routed to: {result['practice_area'].upper()}")
#     print(f"Response: {result['expert_response'][:150]}...\n")

---
## Summary

In this session you learned the foundational skills for working with LLMs programmatically:

1. **API Connections** -- How to initialize the OpenAI client and make chat completion requests.
2. **Tokenization** -- How text is converted to tokens, why token counts matter for cost and context window limits, and how to use `tiktoken` to measure them.
3. **Model Parameters** -- How `temperature`, `top_p`, and `max_tokens` influence the style, creativity, and length of LLM outputs.
4. **System Messages & Personas** -- How the system message acts as a powerful control lever to change LLM behavior without changing the user prompt.
5. **LLM Pipelines** -- How to chain multiple LLM calls together so the output of one step feeds into the next, forming the basis for agentic workflows.
6. **Embeddings & Vector Representations** -- How to generate embeddings using the OpenAI API and compute cosine similarity to measure semantic relatedness between texts.
7. **Embedding Visualization** -- How to use PCA and t-SNE to project high-dimensional embeddings into 2D for visual inspection, and how to build similarity heatmaps.
8. **LLM Reasoning & Agent Behavior** -- How LLMs exhibit emergent capabilities (task decomposition, planning, self-reflection, tool selection) that make them suitable as the core of agentic systems.

**Up Next:** In Session 2, we will dive into advanced prompting techniques -- including few-shot prompting, chain-of-thought reasoning, and structured output generation -- that make LLMs more reliable and useful as the reasoning core of agentic systems.